# 06_T5 — Tradução de inglês para português

**Aluno: Tiago Galhardo Avelar**

Esta é uma cópia resolvida do notebook-modelo da disciplina. O objetivo é ajustar o modelo **PTT5-small** para traduzir do inglês para o português usando pares do **OPUS-100** e medir o resultado com **SacreBLEU**.

O modelo de referência foi escrito para GPU/Google Colab. Para tornar a execução local em CPU verificável, este notebook usa um subconjunto explicitamente reduzido — opção permitida pelo enunciado —, um laço PyTorch próprio e acumulação de gradientes.


## 1. Configuração e reprodutibilidade

São usados 128 pares para treino e 16 para validação, durante cinco épocas. O batch efetivo é 16, obtido acumulando quatro microbatches de quatro exemplos. A validação pequena também reduz o custo da geração autoregressiva exigida pelo SacreBLEU.


In [1]:
import copy
import math
import random
import time

import numpy as np
import sacrebleu
import torch
import transformers
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SEED = 123
MODEL_NAME = "unicamp-dl/ptt5-small-portuguese-vocab"
TRAIN_SAMPLES = 128
VALID_SAMPLES = 16
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS
EPOCHS = 5
LEARNING_RATE = 5e-4
SOURCE_MAX_LENGTH = 48
TARGET_MAX_LENGTH = 48
PREFIX = "translate English to Portuguese: "


def set_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seeds()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo:", DEVICE)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("SacreBLEU:", sacrebleu.__version__)
print("Treino/validação:", TRAIN_SAMPLES, VALID_SAMPLES)
print("Microbatch/acumulação/batch efetivo:", BATCH_SIZE, GRAD_ACCUM_STEPS, EFFECTIVE_BATCH_SIZE)


C:\workspace\fatec\aprendizagem_maquina\.notebook-deps\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo: cpu
PyTorch: 2.14.0+cpu
Transformers: 4.57.3
SacreBLEU: 2.6.0
Treino/validação: 128 16
Microbatch/acumulação/batch efetivo: 4 4 16


## 2. Dataset OPUS-100 inglês–português

O carregamento usa a configuração oficial `en-pt` em modo streaming. São descartados pares vazios; depois os exemplos são embaralhados com semente fixa e divididos sem sobreposição. O conjunto oficial de teste não é consultado.


In [2]:
required = TRAIN_SAMPLES + VALID_SAMPLES
stream = load_dataset(
    "Helsinki-NLP/opus-100",
    "en-pt",
    split="train",
    streaming=True,
)

pairs = []
seen_pairs = set()
for item in stream:
    translation = item["translation"]
    source = translation["en"].strip()
    target = translation["pt"].strip()
    pair = (source, target)
    if source and target and pair not in seen_pairs:
        pairs.append(pair)
        seen_pairs.add(pair)
    if len(pairs) >= required:
        break

random.Random(SEED).shuffle(pairs)
train_pairs = pairs[:TRAIN_SAMPLES]
valid_pairs = pairs[TRAIN_SAMPLES:required]

assert len(train_pairs) == TRAIN_SAMPLES
assert len(valid_pairs) == VALID_SAMPLES
assert not set(train_pairs).intersection(valid_pairs)

print("Pares de treino:", len(train_pairs))
print("Pares de validação:", len(valid_pairs))
for index, (source, target) in enumerate(train_pairs[:3]):
    print(f"{index}: EN: {source}\n   PT: {target}")


Pares de treino: 128
Pares de validação: 16
0: EN: Stand by.
   PT: Estamos sem som.
1: EN: And he's about to get robbed.
   PT: E está prestes a ser roubado.
2: EN: Love, connect grow old, die.
   PT: Amar, casar envelhecer, morrer.


## 3. Tokenizador PTT5 e preparação dos tensores

O prefixo explicita a tarefa de tradução. Entradas e alvos são truncados e preenchidos até 48 tokens. Nos rótulos, posições de padding são substituídas por `-100`, valor ignorado pela entropia cruzada do T5.


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, legacy=True)


def encode_pairs(text_pairs):
    sources = [PREFIX + source for source, _ in text_pairs]
    targets = [target for _, target in text_pairs]

    encoded_sources = tokenizer(
        sources,
        max_length=SOURCE_MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    encoded_targets = tokenizer(
        text_target=targets,
        max_length=TARGET_MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    labels = encoded_targets["input_ids"].clone()
    labels[labels == tokenizer.pad_token_id] = -100

    return TensorDataset(
        encoded_sources["input_ids"],
        encoded_sources["attention_mask"],
        labels,
    )


train_dataset = encode_pairs(train_pairs)
valid_dataset = encode_pairs(valid_pairs)

generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
    num_workers=0,
)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

sample_ids, sample_mask, sample_labels = train_dataset[0]
print("Formas:", sample_ids.shape, sample_mask.shape, sample_labels.shape)
print("Entrada decodificada:", tokenizer.decode(sample_ids, skip_special_tokens=True))
print("Alvo decodificado:", tokenizer.decode(sample_labels[sample_labels != -100], skip_special_tokens=True))


Formas: torch.Size([48]) torch.Size([48]) torch.Size([48])
Entrada decodificada: translate English to Portuguese: Stand by.
Alvo decodificado: Estamos sem som.


## 4. Modelo PTT5-small

O checkpoint possui aproximadamente 60 milhões de parâmetros. Todos permanecem treináveis durante o ajuste fino.


In [4]:
set_seeds()
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(model.__class__.__name__)
print("Parâmetros treináveis:", f"{trainable_parameters:,}")


T5ForConditionalGeneration
Parâmetros treináveis: 60,506,624


## 5. Geração e SacreBLEU antes do ajuste

O corpus BLEU recebe uma lista de hipóteses e uma lista contendo as referências. A configuração é explícita (`lowercase=True`, tokenização `13a`) para tornar o pré-processamento reproduzível.


In [5]:
@torch.no_grad()
def translate_sources(model, sources, batch_size=BATCH_SIZE):
    model.eval()
    hypotheses = []
    for start in range(0, len(sources), batch_size):
        batch_sources = [PREFIX + text for text in sources[start:start + batch_size]]
        encoded = tokenizer(
            batch_sources,
            max_length=SOURCE_MAX_LENGTH,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(DEVICE)
        generated = model.generate(
            **encoded,
            max_new_tokens=TARGET_MAX_LENGTH,
            num_beams=1,
            do_sample=False,
        )
        hypotheses.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    return hypotheses


def calculate_bleu(hypotheses, references):
    return sacrebleu.corpus_bleu(
        hypotheses,
        [references],
        lowercase=True,
        tokenize="13a",
    )


valid_sources = [source for source, _ in valid_pairs]
valid_references = [target for _, target in valid_pairs]
baseline_hypotheses = translate_sources(model, valid_sources)
baseline_bleu = calculate_bleu(baseline_hypotheses, valid_references)

print("SacreBLEU antes do treino:", baseline_bleu)
for source, reference, hypothesis in list(zip(valid_sources, valid_references, baseline_hypotheses))[:3]:
    print("EN:", source)
    print("Referência:", reference)
    print("Hipótese inicial:", hypothesis)
    print()


SacreBLEU antes do treino: BLEU = 1.99 10.5/1.7/1.3/0.7 (BP = 1.000 ratio = 1.949 hyp_len = 191 ref_len = 98)
EN: Is that terrible?
Referência: É horrível?
Hipótese inicial: translate English to Portuguese: Is that terrible?

EN: By carrying them with him he's kept them safe.
Referência: Transportando-os consigo, manteve-os a salvo.
Hipótese inicial: translate English to Portuguese: By carrying them with him he's kept them safe.

EN: That you chose to save me instead of her.
Referência: Que optaste salvar-me em vez dela.
Hipótese inicial: translate English to Portuguese: That you chose to save me instead of her.



## 6. Laço próprio de treinamento com acumulação de gradientes

A loss de cada microbatch é dividida por quatro antes de `backward()`. O otimizador só é atualizado depois de quatro microbatches (ou no último lote da época), reproduzindo batch efetivo 16 sem manter os 16 exemplos simultaneamente na memória.


In [6]:
@torch.no_grad()
def evaluate_loss(model, dataloader):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    for input_ids, attention_mask, labels in dataloader:
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        labels = labels.to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        batch_size = input_ids.size(0)
        total_loss += outputs.loss.item() * batch_size
        total_examples += batch_size
    return total_loss / total_examples


optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
expected_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
optimizer_steps = 0
history = []
best_state = None
best_valid_loss = float("inf")
training_start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    examples_seen = 0
    epoch_start = time.perf_counter()

    for batch_index, (input_ids, attention_mask, labels) in enumerate(train_loader, start=1):
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        (loss / GRAD_ACCUM_STEPS).backward()

        batch_examples = input_ids.size(0)
        running_loss += loss.item() * batch_examples
        examples_seen += batch_examples

        should_step = batch_index % GRAD_ACCUM_STEPS == 0 or batch_index == len(train_loader)
        if should_step:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            optimizer_steps += 1

    train_loss = running_loss / examples_seen
    valid_loss = evaluate_loss(model, valid_loader)
    metrics = {
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_loss": valid_loss,
        "optimizer_steps": optimizer_steps,
        "seconds": time.perf_counter() - epoch_start,
    }
    history.append(metrics)
    print(metrics)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)
training_seconds = time.perf_counter() - training_start
expected_steps = EPOCHS * expected_steps_per_epoch

print("Atualizações esperadas/realizadas:", expected_steps, optimizer_steps)
print(f"Tempo total: {training_seconds:.2f} s")
assert optimizer_steps == expected_steps
assert all(math.isfinite(item["train_loss"]) and math.isfinite(item["valid_loss"]) for item in history)
assert history[-1]["train_loss"] < history[0]["train_loss"]


{'epoch': 1, 'train_loss': 6.986028216779232, 'valid_loss': 3.8918814063072205, 'optimizer_steps': 8, 'seconds': 53.87464599998202}


{'epoch': 2, 'train_loss': 4.061885103583336, 'valid_loss': 3.683222711086273, 'optimizer_steps': 16, 'seconds': 58.60532469989266}


{'epoch': 3, 'train_loss': 3.5130759179592133, 'valid_loss': 3.707707166671753, 'optimizer_steps': 24, 'seconds': 58.20971839991398}


{'epoch': 4, 'train_loss': 3.281608074903488, 'valid_loss': 3.6435930728912354, 'optimizer_steps': 32, 'seconds': 67.51276869990397}


{'epoch': 5, 'train_loss': 2.8891166672110558, 'valid_loss': 3.6748863458633423, 'optimizer_steps': 40, 'seconds': 63.951695000054315}
Atualizações esperadas/realizadas: 40 40
Tempo total: 302.77 s


## 7. Avaliação final com SacreBLEU

O melhor estado segundo a loss de validação é restaurado antes da geração final. Como o experimento usa somente 128 pares em CPU, seu objetivo é demonstrar corretamente o pipeline e registrar uma medição real, não atingir qualidade de produção.


In [7]:
final_hypotheses = translate_sources(model, valid_sources)
final_bleu = calculate_bleu(final_hypotheses, valid_references)

print("SacreBLEU inicial:", baseline_bleu)
print("SacreBLEU final:", final_bleu)
for source, reference, hypothesis in list(zip(valid_sources, valid_references, final_hypotheses))[:5]:
    print("EN:", source)
    print("Referência:", reference)
    print("Tradução:", hypothesis)
    print()

assert len(final_hypotheses) == VALID_SAMPLES
assert math.isfinite(final_bleu.score)


SacreBLEU inicial: BLEU = 1.99 10.5/1.7/1.3/0.7 (BP = 1.000 ratio = 1.949 hyp_len = 191 ref_len = 98)
SacreBLEU final: BLEU = 4.40 24.1/4.2/2.5/1.5 (BP = 1.000 ratio = 1.143 hyp_len = 112 ref_len = 98)
EN: Is that terrible?
Referência: É horrível?
Tradução: Que tal?

EN: By carrying them with him he's kept them safe.
Referência: Transportando-os consigo, manteve-os a salvo.
Tradução: Aí está a ser feito um teste de DNA.

EN: That you chose to save me instead of her.
Referência: Que optaste salvar-me em vez dela.
Tradução: O que é que é o que é.

EN: Why did you tell me that story?
Referência: Por que me contou essa história?
Tradução: ?

EN: Right.
Referência: Pois.
Tradução: Aí está a ser feito um trabalho.



## 8. Evidências consolidadas


In [8]:
summary = {
    "aluno": "Tiago Galhardo Avelar",
    "modelo": MODEL_NAME,
    "dataset": "Helsinki-NLP/opus-100 en-pt",
    "amostras_treino": TRAIN_SAMPLES,
    "amostras_validacao": VALID_SAMPLES,
    "epocas": EPOCHS,
    "microbatch": BATCH_SIZE,
    "acumulacao_gradientes": GRAD_ACCUM_STEPS,
    "batch_efetivo": EFFECTIVE_BATCH_SIZE,
    "passos_otimizador": optimizer_steps,
    "melhor_loss_validacao": best_valid_loss,
    "sacrebleu_inicial": baseline_bleu.score,
    "sacrebleu_final": final_bleu.score,
    "tempo_treino_segundos": training_seconds,
    "dispositivo": str(DEVICE),
}
summary


{'aluno': 'Tiago Galhardo Avelar',
 'modelo': 'unicamp-dl/ptt5-small-portuguese-vocab',
 'dataset': 'Helsinki-NLP/opus-100 en-pt',
 'amostras_treino': 128,
 'amostras_validacao': 16,
 'epocas': 5,
 'microbatch': 4,
 'acumulacao_gradientes': 4,
 'batch_efetivo': 16,
 'passos_otimizador': 40,
 'melhor_loss_validacao': 3.6435930728912354,
 'sacrebleu_inicial': 1.9933986833992305,
 'sacrebleu_final': 4.4028499539088095,
 'tempo_treino_segundos': 302.77219270006754,
 'dispositivo': 'cpu'}

## Ferramentas e referências

- Notebook-modelo da disciplina: [T5 inglês–português](https://colab.research.google.com/drive/1DH05be3mSbRr_xU2y-YHHPuhTr3X4HMF?usp=sharing).
- Carmo et al. (2020): [PTT5: Pretraining and validating the T5 model on Brazilian Portuguese data](https://arxiv.org/abs/2008.09144).
- Modelo: [unicamp-dl/ptt5-small-portuguese-vocab](https://huggingface.co/unicamp-dl/ptt5-small-portuguese-vocab).
- Dataset: [Helsinki-NLP/opus-100](https://huggingface.co/datasets/Helsinki-NLP/opus-100).
- Zhang et al. (2020): [Improving Massively Multilingual Neural Machine Translation and Zero-Shot Translation](https://aclanthology.org/2020.acl-main.148/).
- Post (2018): [A Call for Clarity in Reporting BLEU Scores](https://aclanthology.org/W18-6319/).
- SacreBLEU: [documentação e implementação](https://github.com/mjpost/sacrebleu).
- Hugging Face Transformers: [T5](https://huggingface.co/docs/transformers/model_doc/t5).
- PyTorch: [automatic mixed precision e acumulação manual de gradientes](https://pytorch.org/docs/stable/notes/amp_examples.html#gradient-accumulation).
- Jupyter `nbclient`: [execução programática](https://nbclient.readthedocs.io/).


## Comentários

---
**Comentários:**

Ferramenta: OpenAI Codex (modelo GPT-5)

Utilização: Utilizado como apoio para interpretar o notebook-modelo, adaptar a execução para CPU, desenvolver e revisar o laço de treinamento com acumulação de gradientes, estruturar a avaliação SacreBLEU, documentar referências e validar as evidências executadas.

Ferramenta: Hugging Face Transformers

Utilização: Utilizado para carregar o tokenizador e os pesos pré-treinados do PTT5-small e para realizar a geração autoregressiva.

Ferramenta: Hugging Face Datasets

Utilização: Utilizado para carregar em streaming o subconjunto inglês–português do OPUS-100.

Ferramenta: PyTorch

Utilização: Utilizado para DataLoaders, ajuste fino do modelo, diferenciação automática, acumulação de gradientes, otimização e avaliação da loss.

Ferramenta: SacreBLEU

Utilização: Utilizado para calcular o corpus BLEU com tokenização e lowercase explícitos e reproduzíveis.

Ferramenta: Jupyter Notebook / nbclient

Utilização: Utilizado para executar todas as células e registrar as evidências no arquivo entregue.
---


## Conclusão

O PTT5-small foi ajustado para tradução de inglês para português com pares reais do OPUS-100. O notebook demonstra preparação dos dados, tokenização, laço próprio com acumulação de gradientes, seleção pelo melhor resultado de validação, geração autoregressiva e medição SacreBLEU reproduzível. O tamanho reduzido do subconjunto está documentado e atende à recomendação do enunciado para ambientes com treinamento demorado.
